# Prompt Evaluation

In [1]:
%pip install anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.1/472.1 kB 5.0 MB/s eta 0:00:00


In [2]:
from google.colab import userdata
api_key = userdata.get('ANTHROPIC_API_KEY').strip()

In [3]:
from anthropic import Anthropic
client = Anthropic(api_key=api_key)
model = "claude-haiku-4-5"

In [4]:
# helper functions
def add_user_message(messages, text):
  user_message = {'role': 'user', 'content': text}
  messages.append(user_message)

def add_assistant_message(messages, text):
  assistant_message = {'role': 'assistant', 'content': text}
  messages.append(assistant_message)

def chat(messages, system=None, temperature=1.0, stop_sequences=None):
  params = {
    'model':model,
    'max_tokens':1000,
    'messages':messages,
    'temperature':temperature
  }
  if system:
    params['system'] = system

  if stop_sequences:
    params['stop_sequences'] = stop_sequences

  message = client.messages.create(**params)
  return message.content[0].text

In [5]:
# eval dataset generation
import json

def generate_dataset():
  prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

  messages = []
  add_user_message(messages, prompt)
  add_assistant_message(messages, '```json')
  text = chat(messages, stop_sequences=['```'])
  return json.loads(text)

In [6]:
# save dataset file
dataset = generate_dataset()

with open('dataset.json', 'w') as f:
  json.dump(dataset, f, indent=2)

In [9]:
# model-based grading
def grade_by_model(test_case, output):
    eval_prompt = f"""
You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{output}
</solution>

Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "score": A number between 1-10

Respond with JSON. Keep your response concise and direct.
Example response shape:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
}}
    """

    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

In [17]:
# more helper functions
from statistics import mean

def run_prompt(test_case):
  prompt =f"""
    Please solve the following task:
    {test_case['task']}
    """

  messages = []
  add_user_message(messages, prompt)
  output = chat(messages)
  return output

def run_test_case(test_case):
  output = run_prompt(test_case)
  model_grade = grade_by_model(test_case, output)
  score = model_grade['score']
  reasoning = model_grade['reasoning']

  return {
      'output': output,
      'test_case': test_case,
      'score': score,
      'reasoning': reasoning,
  }

def run_eval(dataset):
  results = []

  for test_case in dataset:
    result = run_test_case(test_case)
    results.append(result)

  average_score = mean([result['score'] for result in results])
  print(f'Average Score: {average_score}')

  return results

In [18]:
with open('dataset.json', 'r') as f:
  dataset = json.load(f)
results = run_eval(dataset)

Average Score: 7.666666666666667


In [19]:
print(json.dumps(results, indent=2))

[
  {
    "output": "# AWS Account ID Extractor from ARN\n\n```python\ndef extract_account_id_from_arn(arn: str) -> str | None:\n    \"\"\"\n    Extracts the AWS account ID from an ARN (Amazon Resource Name) string.\n    \n    Args:\n        arn: An AWS ARN string in the format:\n             arn:partition:service:region:account-id:resource-type/resource-id\n    \n    Returns:\n        The 12-digit AWS account ID if present, or None if the ARN doesn't contain one.\n    \n    Examples:\n        >>> extract_account_id_from_arn('arn:aws:iam::123456789012:role/MyRole')\n        '123456789012'\n        \n        >>> extract_account_id_from_arn('arn:aws:s3:::my-bucket')\n        None\n        \n        >>> extract_account_id_from_arn('arn:aws:ec2:us-east-1:123456789012:instance/i-1234567890abcdef0')\n        '123456789012'\n    \"\"\"\n    if not arn or not isinstance(arn, str):\n        return None\n    \n    # Split ARN by colons\n    parts = arn.split(':')\n    \n    # ARN format: arn:par